In [2]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt


In [2]:
ds = xr.open_dataset("./data/ice_conc_nh_ease2-250_cdr-v3p0-amsr_200206011200.nc")


In [3]:
ds

<xarray.Dataset> Size: 10MB
Dimensions:                         (time: 1, nv: 2, xc: 432, yc: 432)
Coordinates:
  * time                            (time) datetime64[ns] 8B 2002-06-01T12:00:00
  * xc                              (xc) float64 3kB -5.388e+03 ... 5.388e+03
  * yc                              (yc) float64 3kB 5.388e+03 ... -5.388e+03
    lat                             (yc, xc) float32 746kB ...
    lon                             (yc, xc) float32 746kB ...
Dimensions without coordinates: nv
Data variables:
    Lambert_Azimuthal_Grid          int32 4B ...
    time_bnds                       (time, nv) datetime64[ns] 16B ...
    ice_conc                        (time, yc, xc) float64 1MB ...
    raw_ice_conc_values             (time, yc, xc) float64 1MB ...
    total_standard_uncertainty      (time, yc, xc) float64 1MB ...
    smearing_standard_uncertainty   (time, yc, xc) float64 1MB ...
    algorithm_standard_uncertainty  (time, yc, xc) float64 1MB ...
    status_flag                     (time, yc, xc) float32 746kB ...
Attributes: (12/43)
    title:                     Sea Ice Concentration Climate Data Record Vers...
    summary:                   This climate data record of sea ice concentrat...
    topiccategory:             Oceans ClimatologyMeteorologyAtmosphere
    geospatial_lat_min:        16.62393
    geospatial_lat_max:        90.0
    geospatial_lon_min:        -180.0
    ...                        ...
    algorithm:                 SICCI3LF (19V, 37V, 37H)
    references:                Product User Manual v3 (2022),Algorithm Theore...
    product_id:                osi-458
    contributor_name:          Thomas Lavergne,Atle Soerensen,Rasmus Tonboe,C...
    contributor_role:          PrincipalInvestigator,author,author,author,aut...
    product_status:            released

In [ ]:
# load entire data folder in memory, # this is not recommended for large datasets
# and do this for all time series
dt = xr.open_mfdataset("./data/ice_conc_nh_ease2-250_cdr-v3p0-amsr_*.nc",
                          combine='by_coords',
                          parallel=True,
                          chunks={'time': 1})
# print the dataset
print(dt)

In [3]:
# load entire data folder in memory, # this is not recommended for large datasets
# and do this for all time series
dt = xr.open_mfdataset("./data/ice_conc_nh_ease2-250_cdr-v3p0-amsr_*.nc",
                          combine='by_coords',
                          chunks={'time': 1})
# print the dataset
print(dt)

<xarray.Dataset> Size: 231MB
Dimensions:                         (time: 28, nv: 2, yc: 432, xc: 432)
Coordinates:
  * time                            (time) datetime64[ns] 224B 2002-06-01T12:...
  * xc                              (xc) float64 3kB -5.388e+03 ... 5.388e+03
  * yc                              (yc) float64 3kB 5.388e+03 ... -5.388e+03
    lat                             (yc, xc) float32 746kB dask.array<chunksize=(432, 432), meta=np.ndarray>
    lon                             (yc, xc) float32 746kB dask.array<chunksize=(432, 432), meta=np.ndarray>
Dimensions without coordinates: nv
Data variables:
    Lambert_Azimuthal_Grid          (time) int32 112B -2147483647 ... -214748...
    time_bnds                       (time, nv) datetime64[ns] 448B dask.array<chunksize=(1, 2), meta=np.ndarray>
    ice_conc                        (time, yc, xc) float64 42MB dask.array<chunksize=(1, 432, 432), meta=np.ndarray>
    raw_ice_conc_values             (time, yc, xc) float64 42MB dask.

In [6]:
print(dt["ice_conc"].isel(yc=430, xc=430).to_numpy())

[nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan]


In [ ]:
ice_conc_interpolated = dt["ice_conc"].interpolate_na(
    dim="time", method="linear"
)

ValueError: dimension time on 0th function argument to apply_ufunc with dask='parallelized' consists of multiple chunks, but is also a core dimension. To fix, either rechunk into a single array chunk along this dimension, i.e., ``.chunk(dict(time=-1))``, or pass ``allow_rechunk=True`` in ``dask_gufunc_kwargs`` but beware that this may significantly increase memory usage.

In [9]:
ice_conc_rechunked = dt["ice_conc"].chunk(
    {"time": -1, "yc": "auto", "xc": "auto"}
)

In [10]:
ice_conc_interpolated = ice_conc_rechunked.interpolate_na(
    dim="time", method="linear"
)

In [11]:
dt["ice_conc_interpolated"] = ice_conc_interpolated

In [12]:
print(print(dt["ice_conc"].isel(yc=430, xc=430).to_numpy()))
print(print(dt["ice_conc_interpolated"].isel(yc=430, xc=430).to_numpy()))

[nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan]
None
[nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan]
None


In [25]:
original_nans = dt['ice_conc'].isnull()
# print(original_nans.values)

filled_by_interpolation = original_nans & ~dt['ice_conc_interpolated'].isnull()
print(filled_by_interpolation.values)

[[[False False False ... False False False]
  [False False False ... False False False]
  [False False False ... False False False]
  ...
  [False False False ... False False False]
  [False False False ... False False False]
  [False False False ... False False False]]

 [[False False False ... False False False]
  [False False False ... False False False]
  [False False False ... False False False]
  ...
  [False False False ... False False False]
  [False False False ... False False False]
  [False False False ... False False False]]

 [[False False False ... False False False]
  [False False False ... False False False]
  [False False False ... False False False]
  ...
  [False False False ... False False False]
  [False False False ... False False False]
  [False False False ... False False False]]

 ...

 [[False False False ... False False False]
  [False False False ... False False False]
  [False False False ... False False False]
  ...
  [False False False ... False False Fal

In [26]:
if filled_by_interpolation.any():
    print("There are some values that were filled by interpolation.")

In [29]:
import pandas as pd
from scipy.interpolate import griddata

In [28]:
!pip install scipy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 37.4 MB/s eta 0:00:0000:0100:01


In [30]:
ice_conc_spatial = dt["ice_conc_interpolated"].copy()

In [31]:
xc_grid, yc_grid = np.meshgrid(ice_conc_spatial['xc'].values, ice_conc_spatial['yc'].values)

In [35]:
print(ice_conc_spatial['yc'])

<xarray.DataArray 'yc' (yc: 432)> Size: 3kB
array([ 5387.5,  5362.5,  5337.5, ..., -5337.5, -5362.5, -5387.5], shape=(432,))
Coordinates:
  * yc       (yc) float64 3kB 5.388e+03 5.362e+03 ... -5.362e+03 -5.388e+03
Attributes:
    units:          km
    long_name:      y coordinate of projection (northings)
    standard_name:  projection_y_coordinate


In [36]:
current_slice_data = ice_conc_spatial.isel(time=0).values
print(current_slice_data)

[[ 0.  0.  0. ...  0.  0.  0.]
 [ 0.  0.  0. ...  0.  0.  0.]
 [ 0.  0.  0. ...  0.  0.  0.]
 ...
 [ 0.  0.  0. ... nan nan nan]
 [ 0.  0.  0. ... nan nan nan]
 [ 0.  0.  0. ... nan nan nan]]


In [37]:
valid_mask = ~np.isnan(current_slice_data)

In [58]:
valid_mask
coords = np.vstack((yc_grid[valid_mask], xc_grid[valid_mask])).T
coords
value = current_slice_data[valid_mask]

In [59]:
value

array([0., 0., 0., ..., 0., 0., 0.], shape=(97777,))

In [60]:
nan_coords = np.vstack((yc_grid[~valid_mask], xc_grid[~valid_mask])).T

In [61]:
nan_coords.size,coords.size

(177694, 195554)

In [62]:
if len(value)>0 and len(nan_coords)>0:
    interpolated_values = griddata(points=coords, values=value, xi=nan_coords, method='nearest', fill_value=np.nan)
    current_slice_data[~valid_mask] = interpolated_values
    ice_conc_spatial[0,:,:] = current_slice_data

print(ice_conc_spatial.isel(time=0).values)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [64]:
print(ice_conc_spatial.isnull().sum().values)

2398869
